# Wireless Networking and Mobile Computing

### Prerequisites

This Jupyter Notebook has been tested with Visual Studio Code, running in a local Python environment.

**Installs**

In [ ]:
# --- Installs: ensure required packages are available ---
%pip install ipywidgets collections --quiet

**Imports**

In [ ]:
# --- Imports: libraries used in this notebook ---
import ipywidgets as widgets
from IPython.display import display
from collections import Counter
import math

### 1. Introduction

#### 1.1 Source Coding
**Source coding** is the process of converting data (or a signal) into a form that can be efficiently stored or transmitted. 
The primary goal is to reduce redundancy (or unnecessary information) in the data representation while still maintaining essential information.

In mobile computing, devices exchange large amounts of data (images, audio, video, text) over constrained wireless channels.
Efficient compression ensures:

- Less data to transmit ⟶ faster communication
- Reduced energy consumption ⟶ longer battery life
- Lower bandwidth usage ⟶ less interference

This notebook introduces source coding (compressing information at the source) and explores fundamental concepts like entropy, lossless vs. lossy compression, and practical algorithms.


### 1.2 Types of Compression  

Compression techniques can be broadly divided into two categories:  

**Lossless Compression**  
- Preserves the original data exactly, with no information loss.  
- Essential when accuracy is critical (e.g., financial transactions, executable code, medical records).  
- Common formats: **ZIP**, **PNG**, **FLAC**.  

**Lossy Compression**  
- Sacrifices some data fidelity to achieve higher compression ratios.  
- Appropriate when perfect reconstruction is not required, such as in human-perceived media.  
- Common formats: **JPEG**, **MP3**, **MP4**.


 
### 1.3 Why Compression Matters  

Modern data can be extremely large. For example, an uncompressed HD video may exceed 1 GB per minute, which quickly becomes impractical to store or transmit. Wireless networks also have limited bandwidth, and every bit transmitted consumes battery power on mobile devices. Storage is another constraint: without compression, even basic applications like saving photos or movies would quickly overwhelm available space.  

Compression is what makes many everyday applications possible. Streaming services, cloud backups, and mobile messaging all rely on reducing the size of data so it can be shared quickly and efficiently. A two-hour HD film, which would take about 25 GB in uncompressed form, fits onto a Blu-ray disc only because advanced video compression standards such as H.264 or H.265 shrink it dramatically while still keeping the quality watchable.  

Different kinds of data call for different compression methods. Text may use Huffman or arithmetic coding, audio is often compressed with MP3 or AAC, images with PNG, JPEG, or WebP, and video with formats such as H.264 or AV1. The common thread in all these approaches is the same: they exploit patterns and redundancy in the data to reduce its size - sometimes exactly (lossless), sometimes approximately (lossy).  



### 2. Huffman Encoding  

So far, we have seen that the entropy of a source gives us a theoretical limit on how much we can compress data. The next question is: how do we design an actual coding scheme that gets close to this limit in practice?  

One of the most widely used techniques is **Huffman encoding**, introduced by David Huffman in 1952. It is a **lossless compression algorithm** that assigns shorter binary codes to frequent symbols and longer codes to rare ones. The result is a **prefix-free code**: no symbol’s code is the prefix of another, so decoding is always unambiguous.  

The core idea is simple:  
1. Count how often each symbol occurs (its frequency).  
2. Build a binary tree by repeatedly merging the two least frequent symbols.  
3. Assign binary codes to symbols by following paths in the tree (left = 0, right = 1).  

This way, the average number of bits per symbol approaches the source entropy. Huffman coding is the foundation of many compression formats, including **ZIP**, **PNG**, and parts of **MP3**.  


#### 2.1 Visual Walkthrough  

To make the process concrete, let’s look at this GIF, which shows Huffman coding applied to a random text :  

<center>
<img src="../../Admin/picture/Huffman_huff_demo.gif" 
     alt="Huffman Coding Demonstration" 
     width="60%" 
     style="filter: grayscale(100%);">
<p style="text-align: center; font-size: 0.9em;">
Source: <a href="https://upload.wikimedia.org/wikipedia/commons/a/ac/Huffman_huff_demo.gif">Wikimedia Commons</a> · License: CC BY-SA 3.0
</p>
</center>
 

Watch carefully:  
- At the start, each character is listed with its frequency.  
- The algorithm repeatedly merges the two least frequent nodes into a new parent node.  
- When only one node remains (the root), the tree is complete.  
- Traversing the tree gives us the codes: left edge = `0`, right edge = `1`.  

The final output assigns short codes to frequent letters (like `a` and `t`) and longer codes to rare ones. This reduces the total size of the encoded text compared to using a fixed-length scheme such as ASCII.  



#### 2.2 Why Huffman Coding Works  

The power of Huffman coding comes from exploiting **non-uniform symbol frequencies**. If all symbols were equally likely, no compression would be possible. But real-world data is far from uniform:  
- In English text, letters like `e`, `t`, and `a` are far more common than `z` or `q`.  
- In sensor data, some values repeat much more often than others.  

By matching code lengths to probabilities, Huffman coding makes efficient use of bits and gets us close to the entropy bound.  


### 3. Compression

#### 3.1 Lossy Compression: JPEG  

To see lossy compression in action, we’ll take a sample image from `scikit-image` and save it at different JPEG quality levels. This will let us compare how reducing quality affects both the file size and the visible details of the image.  


In [ ]:

from skimage import data
from PIL import Image
import io
import matplotlib.pyplot as plt

# Load a sample image (astronaut)
cat = data.chelsea()
img = Image.fromarray(cat)

qualities = [95, 70, 40, 10, 1]
compressed_images = []
sizes_kb = []

for q in qualities:
    # Save into a memory buffer instead of disk
    buffer = io.BytesIO()
    img.save(buffer, format="JPEG", quality=q)
    size_kb = len(buffer.getvalue()) / 1024
    sizes_kb.append(size_kb)

    # Reload the image from buffer for display
    buffer.seek(0)
    compressed_images.append(Image.open(buffer))

# Plot images side by side
fig, axes = plt.subplots(1, len(qualities), figsize=(15, 5))
for ax, q, im, s in zip(axes, qualities, compressed_images, sizes_kb):
    ax.imshow(im)
    ax.set_title(f"Q={q}\n{int(s)} KB")
    ax.axis("off")

plt.show()


#### 3.2 Trade-Offs in Compression  

In mobile and wireless communication, resources such as **bandwidth**, **storage**, and **battery power** are limited. Compression helps reduce the amount of data that must be stored or transmitted, but it also introduces important trade-offs:  

- **Lossless vs. Lossy**  
  Lossless methods preserve every bit of information but usually achieve lower compression ratios. Lossy methods achieve much stronger compression but may reduce quality or introduce distortion.  

- **Compression Ratio vs. Computation**  
  More advanced algorithms can shrink data more effectively, but they also require more processing power. On mobile devices, this can increase energy consumption and reduce battery life.  

- **Latency vs. Quality**  
  Real-time applications like video calls or VoIP need low latency. Highly compressed streams save bandwidth, but aggressive compression can reduce visual or audio quality.  

Because of these trade-offs, the “best” compression method depends on the application requirements:  
- For bank transactions or medical data, accuracy matters most -> use lossless.  
- For streaming video, efficiency and smooth playback are critical -> use lossy.  
- For real-time communication, low delay is essential -> lightweight codecs are preferred.  